In [25]:
!pip install transformers datasets accelerate evaluate torch gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [32]:
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from datasets import Dataset, load_dataset
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# **Load and Explore Dataset**

In [52]:
dataset = load_dataset("nbertagnolli/counsel-chat", split="train")
df = dataset.to_pandas()
print(f"Total examples: {len(df)}")

Repo card metadata block was not found. Setting CardData to empty.


Total examples: 2775


In [53]:
df.head()

,questionID,questionTitle,questionText,questionLink,topic,therapistInfo,therapistURL,answerText,upvotes,views
0,0,Do I have too many issues for counseling?,I have so many issues to address. I have a his...,https://counselchat.com/questions/do-i-have-to...,depression,Jennifer MolinariHypnotherapist & Licensed Cou...,https://counselchat.com/therapists/jennifer-mo...,It is very common for people to have multiple ...,3,1971
1,0,Do I have too many issues for counseling?,I have so many issues to address. I have a his...,https://counselchat.com/questions/do-i-have-to...,depression,"Jason Lynch, MS, LMHC, LCAC, ADSIndividual & C...",https://counselchat.com/therapists/jason-lynch...,"I've never heard of someone having ""too many i...",2,386
2,0,Do I have too many issues for counseling?,I have so many issues to address. I have a his...,https://counselchat.com/questions/do-i-have-to...,depression,Shakeeta TorresFaith Based Mental Health Couns...,https://counselchat.com/therapists/shakeeta-to...,Absolutely not. I strongly recommending worki...,2,3071
3,0,Do I have too many issues for counseling?,I have so many issues to address. I have a his...,https://counselchat.com/questions/do-i-have-to...,depression,"Noorayne ChevalierMA, RP, CCC, CCAC, LLP (Mich...",https://counselchat.com/therapists/noorayne-ch...,Let me start by saying there are never too man...,2,2643
4,0,Do I have too many issues for counseling?,I have so many issues to address. I have a his...,https://counselchat.com/questions/do-i-have-to...,depression,"Toni Teixeira, LCSWYour road to healing begins...",https://counselchat.com/therapists/toni-teixei...,I just want to acknowledge you for the courage...,1,256


In [54]:
df = df[['questionText', 'answerText', 'topic']]
df.dropna(inplace=True)
print(f"After dropping NA: {len(df)}")

After dropping NA: 2612


# **Data Preprocessing**

In [55]:
def format_example(row):
    return f"User: {row['questionText']}\nBot: {row['answerText']}"

In [56]:
df['text'] = df.apply(format_example, axis=1)

In [57]:
hf_dataset = Dataset.from_pandas(df[['text']])

In [58]:
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = hf_dataset["train"]
valid_dataset = hf_dataset["test"]

In [59]:
print(f"Train size: {len(train_dataset)}, Eval size: {len(valid_dataset)}")
print("\nSample:")
print(train_dataset[0]["text"])

Train size: 2350, Eval size: 262

Sample:
User: My boyfriend is in recovery from drug addiction. We recently got into a fight and he has become very distant. I don't know what to do to fix the relationship.
Bot: My empathy goes out to you. Relationships are tough enough and i'm sure your partner being distant has been upsetting for you. You are confused and want to mend the relationship with him, but are finding it so hard to do. Relationships require both people to work consistently in order to keep them happier, but if it is only you that is actively trying to repair the relationship, it can be emotionally draining. Perhaps expressing you feel using I, you statements can be beneficial. Do not accuse him or use "you" statements like "you are so distant"  or "you don't even care".Start out sharing your feelings by saying "I feel sad when you don't return my phone calls" or "I feel confused about our relationship ever since you have been keeping to yourself". He may have a valid point, 

# **Tokenization**

In [60]:
model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [61]:
tokenizer.pad_token = tokenizer.eos_token

In [62]:
def tokenize_function(examples):
    """
    Tokenize text examples with truncation and padding.
    """
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

In [64]:
tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_valid = valid_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
print(f"Tokenized training samples: {len(tokenized_train)}")
print(f"Tokenized validation samples: {len(tokenized_valid)}")

Map:   0%|          | 0/2350 [00:00<?, ? examples/s]

Map:   0%|          | 0/262 [00:00<?, ? examples/s]

Tokenized training samples: 2350
Tokenized validation samples: 262


# **Load Model**

In [65]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [66]:
model.resize_token_embeddings(len(tokenizer))
print(f"Model parameters: {model.num_parameters():,}")

Model parameters: 81,912,576


# **Training Configuration**

In [67]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM
)

In [71]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./mental_health_chatbot",
    num_train_epochs=3,                  # Number of training epochs
    per_device_train_batch_size=8,       # Batch size per device
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,       # Simulate larger batch size
    eval_strategy="steps",
    eval_steps=500,                      # Evaluate every 500 steps
    save_steps=1000,
    logging_steps=100,
    learning_rate=5e-5,                  # Learning rate
    weight_decay=0.01,
    fp16=True,                           # Mixed precision
    push_to_hub=False,                   # Set True to upload to Hugging Face Hub
    report_to="none"                     # Disable external logging
)
print("Training arguments configured successfully.")

Training arguments configured successfully.


# **Initialize Trainer and Start Training**

In [72]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
)

In [73]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=441, training_loss=2.8476556963931405, metrics={'train_runtime': 88.0536, 'train_samples_per_second': 80.065, 'train_steps_per_second': 5.008, 'total_flos': 230267761459200.0, 'train_loss': 2.8476556963931405, 'epoch': 3.0})

In [74]:
model.save_pretrained("./mental_health_chatbot_final")
tokenizer.save_pretrained("./mental_health_chatbot_final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mental_health_chatbot_final/tokenizer_config.json',
 './mental_health_chatbot_final/tokenizer.json')

In [75]:
print("Model saved to './mental_health_chatbot_final'")

Model saved to './mental_health_chatbot_final'


# **Load Fine-Tuned Model for Inference**

In [76]:
from transformers import pipeline

In [77]:
model_path = "./mental_health_chatbot_final"
fine_tuned_model = AutoModelForCausalLM.from_pretrained(model_path)
fine_tuned_tokenizer = AutoTokenizer.from_pretrained(model_path)
fine_tuned_tokenizer.pad_token = fine_tuned_tokenizer.eos_token

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

# **Response Generation Function**

In [78]:
def generate_response(user_input, max_length=80, temperature=0.9, top_p=0.9):
    """
    Generate an empathetic response for the given user input.

    Args:
        user_input (str): User's message
        max_length (int): Maximum length of generated response
        temperature (float): Controls randomness (higher = more creative)
        top_p (float): Nucleus sampling parameter

    Returns:
        str: Generated bot response
    """
    # Format input as conversation
    prompt = f"User: {user_input}\nBot:"

    # Tokenize
    inputs = fine_tuned_tokenizer.encode(prompt, return_tensors="pt")

    # Generate response
    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            inputs,
            max_length=len(inputs[0]) + max_length,
            temperature=temperature,
            do_sample=True,
            top_p=top_p,
            pad_token_id=fine_tuned_tokenizer.eos_token_id,
            repetition_penalty=1.2  # Reduce repetition
        )

    # Decode and extract bot's response
    full_response = fine_tuned_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the part after "Bot:"
    if "Bot:" in full_response:
        response = full_response.split("Bot:")[-1].strip()
    else:
        response = full_response.strip()

    return response

# **Test the Chatbot**

In [79]:
test_inputs = [
    "I've been feeling really anxious about my job interview tomorrow.",
    "I had a fight with my best friend and I feel awful.",
    "I feel so lonely since moving to a new city.",
    "I'm proud of myself for finishing a big project.",
    "I've been sad all week and I don't know why."
]

In [80]:
for user_input in test_inputs:
    print(f"\n😟 User: {user_input}")
    response = generate_response(user_input)
    print(f"🤗 Bot: {response}")
    print("-" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



😟 User: I've been feeling really anxious about my job interview tomorrow.
🤗 Bot: It is not uncommon for a person to feel nervous and upset at the beginning of their interviews because it may indicate that something important has happened or if they are in fact working towards an agreement with someone else (e-mailing them). Sometimes, however...it appears as though everyone involved will get into some sort from being so relaxed when your question comes up - like you did on previous ones!
------------------------------------------------------------

😟 User: I had a fight with my best friend and I feel awful.
🤗 Bot: If you’ve been fighting for someone, what is the most painful process to have?  There are so many ways in which we can learn about ourselves that our feelings aren't being addressed by others at times (such as "I'm not going away from her anymore"). The therapy consists of both learning more than just looking up or moving past things around it--it's really hard to imagine
--

# **Gradio Web Interface**

In [81]:
def chat_interface(message, history):
    """
    Gradio chat function that maintains conversation history.
    """
    response = generate_response(message)
    return response


In [82]:
demo = gr.ChatInterface(
    fn=chat_interface,
    title="Mental Health Support Bot",
    description="""A chatbot fine-tuned to provide empathetic and supportive responses for mental health concerns.

    **Note:** This is a demonstration tool only. Not a substitute for professional mental health care.
    If you're in crisis, please contact a mental health professional or emergency services.""",
    theme="soft"
)

In [83]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f99d6fb4a26a4ab8f4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# **Conclusion**

**Key Insights:**
- DistilGPT2 was successfully fine-tuned on the EmpatheticDialogues dataset
- The model learns to generate contextually appropriate, empathetic responses
- The model shows improved empathy compared to base DistilGPT2
